# Benchmark bottom-identity validation

This notebook measures reconciliation with the optimized bottom-identity validation in `HierarchicalReconciliation.reconcile`.

The validation always runs for a DataFrame-based `S_df`; it only materializes the bottom `n_bottom x n_bottom` block. Sparse `SMatrix` outputs from `aggregate(..., sparse_s=True)` carry verified provenance and cache successful checks for repeated reconciliation.

In [1]:
import inspect
import sys
import time
from pathlib import Path

repo_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "hierarchicalforecast" / "core.py").exists()
)
sys.path.insert(0, str(repo_root))
for module_name in list(sys.modules):
    if module_name == "hierarchicalforecast" or module_name.startswith("hierarchicalforecast."):
        del sys.modules[module_name]

import numpy as np
import polars as pl

from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import TopDown

print(f"Using hierarchicalforecast from: {inspect.getfile(HierarchicalReconciliation)}")
print(inspect.signature(HierarchicalReconciliation.reconcile))
assert "skip_bottom_identity_check" not in inspect.signature(HierarchicalReconciliation.reconcile).parameters

Using hierarchicalforecast from: /Users/janrathfelder/Documents/data_science/GitHub/hierarchicalforecast/hierarchicalforecast/core.py
(self, Y_hat_df: Union[ForwardRef('DataFrame[Any]'), ForwardRef('LazyFrame[Any]')], tags: dict[str, numpy.ndarray], S_df: 'Frame | SMatrix' = None, Y_df: Union[ForwardRef('DataFrame[Any]'), ForwardRef('LazyFrame[Any]'), NoneType] = None, level: list[int] | None = None, intervals_method: str = 'normality', num_samples: int = -1, seed: int = 0, is_balanced: bool = False, id_col: str = 'unique_id', time_col: str = 'ds', target_col: str = 'y', id_time_col: str = 'temporal_id', temporal: bool = False, diagnostics: bool = False, diagnostics_atol: float = 1e-06) -> ~FrameT


In [2]:
import utilsforecast, narwhals
print("utilsforecast", utilsforecast.__version__)
print("narwhals", narwhals.__version__)

utilsforecast 0.2.14
narwhals 2.24.0


## Build a strict hierarchy

The generated `S_df` has aggregate rows first and bottom rows last. The final `n_bottom x n_bottom` block is an identity matrix, matching the output contract expected from `aggregate`.

In [3]:
# Increase these if the timing difference is too small on your machine.
n_groups = 80
bottom_per_group = 25
horizon = 8
seed = 0

rng = np.random.default_rng(seed)
n_bottom = n_groups * bottom_per_group

bottom_ids = np.array([f"bottom_{i:05d}" for i in range(n_bottom)])
group_ids = np.array([f"group_{i:04d}" for i in range(n_groups)])
all_ids = np.concatenate([["total"], group_ids, bottom_ids])

S = np.zeros((1 + n_groups + n_bottom, n_bottom), dtype=np.float64)
S[0, :] = 1.0
for group_idx in range(n_groups):
    start = group_idx * bottom_per_group
    stop = start + bottom_per_group
    S[1 + group_idx, start:stop] = 1.0
S[1 + n_groups :, :] = np.eye(n_bottom)

S_df = pl.DataFrame(
    {"unique_id": all_ids, **{bottom_id: S[:, i] for i, bottom_id in enumerate(bottom_ids)}}
)

bottom_fcsts = rng.uniform(10, 100, size=(n_bottom, horizon))
base_fcsts = S @ bottom_fcsts

Y_hat_df = pl.DataFrame(
    {
        "unique_id": np.repeat(all_ids, horizon),
        "ds": np.tile(np.arange(horizon), len(all_ids)),
        "model": base_fcsts.reshape(-1),
    }
)

tags = {
    "total": np.array(["total"]),
    "groups": group_ids,
    "bottom": bottom_ids,
}

print(f"n_series={len(all_ids):,}, n_bottom={n_bottom:,}, horizon={horizon}")
print(f"S_df shape={S_df.shape}, Y_hat_df shape={Y_hat_df.shape}")

n_series=2,081, n_bottom=2,000, horizon=8
S_df shape=(2081, 2001), Y_hat_df shape=(16648, 3)


## Sanity check the condition being validated

The bottom `n_bottom x n_bottom` block must be an identity matrix. The implementation materializes and checks only this block.

In [4]:
bottom_block = S[-n_bottom:, :]
assert np.allclose(bottom_block, np.eye(n_bottom))
print("Bottom block is identity.")

Bottom block is identity.


## Time reconciliation with optimized bottom-identity validation

In [7]:
def time_reconcile(repeats: int = 20):
    timings = []
    last_result = None
    for _ in range(repeats):
        hrec = HierarchicalReconciliation([TopDown(method="forecast_proportions")])
        start = time.perf_counter()
        last_result = hrec.reconcile(
            Y_hat_df=Y_hat_df,
            S_df=S_df,
            tags=tags,
        )
        timings.append(time.perf_counter() - start)
    return np.array(timings), last_result

timings, result = time_reconcile()

print(f"optimized validation: {timings.round(4)} seconds, mean={timings.mean():.4f}")

optimized validation: [0.862  0.7388 0.7287 0.68   0.675  0.7615 0.6793 0.8235 0.7316 0.6826
 0.6677 0.6786 0.7135 0.673  0.6597 0.6526 0.6714 0.667  0.6764 0.6803] seconds, mean=0.7052


## Verify outputs match

In [8]:
result_col = "model/TopDown_method-forecast_proportions"

assert result[result_col].len() == len(Y_hat_df)
assert np.isfinite(result[result_col].to_numpy()).all()
print("Reconciliation completed with finite forecasts for every input row.")

Reconciliation completed with finite forecasts for every input row.


## Expected production use

```python
Y_df, S_df, tags = aggregate(df, spec)

hrec.reconcile(
    Y_hat_df=Y_hat_df,
    S_df=S_df,
    tags=tags,
)
```